<a href="https://colab.research.google.com/github/rkk7452/OVRA-PyBDSF-73MHz/blob/main/Ryan_extract_sources_with_bdsf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Import necessary packages
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.wcs import WCS
from astropy.visualization import ZScaleInterval, ImageNormalize
import os
import pandas as pd

In [2]:
# Install pyBDSF (https://github.com/lofar-astron/pybdsf)
%pip install bdsf

# Install a utility that we'll use to uncompress the .fits files
# Compressed files can be imaged but we can't run BDSF on them
!apt install -y libcfitsio-bin

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libcfitsio-bin is already the newest version (4.0.0-1).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.


In [3]:
import bdsf

In [5]:
folder_path = "/content/drive/MyDrive/Caltech SRC '26/OVRO-LWA Data/"
ryan_folder_path = "/content/drive/MyDrive/Caltech SRC '26/"
file_name = "73MHz-Clean-Snapshot-20250507_095214-image.fits"

In [ ]:
#TO DOs:

#create a list of all unique file names in this folder at 73MHz.
#Save the part between Clean-Snapshot and -image.fits in the file name into a list
#then iterate through the list and run the pybdsf and create a dataframe with the name of the file
#then write an algorithm to compare these different dataframes


In [6]:
# Needed if running on Google Colab only
# Gives access to data stored on Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Confirm files on Google Drive
datafiles = os.listdir(folder_path)  # Change path to whatever folder holds your data
print(f"Datafile names: {datafiles}")  # Confirm that the datafiles were found

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Datafile names: ['73MHz-Clean-Snapshot-20250507_095214-image.fits.fz', '73MHz-Clean-Snapshot-20250507_095225-image.fits.fz', '73MHz-Clean-Snapshot-20250507_095235-image.fits.fz', '73MHz-Clean-Snapshot-20250507_095245-image.fits.fz', '73MHz-Clean-Snapshot-20250507_095255-image.fits.fz', '73MHz-Clean-Snapshot-20250507_095305-image.fits.fz', '73MHz-Clean-Snapshot-20250507_095325-image.fits.fz', '73MHz-Clean-Snapshot-20250507_095315-image.fits.fz', '73MHz-Clean-Snapshot-20250507_095346-image.fits.fz', '73MHz-Clean-Snapshot-20250507_095335-image.fits.fz', '73MHz-Clean-Snapshot-20250507_095356-image.fits.fz', '73MHz-Clean-Snapshot-20250507_095416-image.fits.fz', '73MHz-Clean-Snapshot-20250507_095406-image.fits.fz', '73MHz-Clean-Snapshot-20250507_095426-image.fits.fz', '73MHz-Clean-Snapshot-20250507_095436-image.fits.fz', '73MHz-Clean-Snapshot-20250507_095446-image.

In [14]:

# Initialize the master dataframe
combined_df = pd.DataFrame(columns=['ra', 'dec', 'total_flux', 'file_name'])
combined_csv_path = f"{ryan_folder_path}combined_sources_73MHz.csv"

# creating a list of file names at 73MHz
file_list_73 = [item for item in datafiles if ("73MHz-Clean-Snapshot" in item and item.endswith("-image.fits.fz"))]
file_list_73 = [item.removesuffix(".fz") for item in file_list_73]
clean_file_list_73 = list(map(lambda x: f"{x.split('-')[0]}-{x.split('-')[3]}", file_list_73)) #file names with just frequency, date, and time (e.g. '73MHz-20250507_095214')
print(file_list_73)

['73MHz-Clean-Snapshot-20250507_095214-image.fits', '73MHz-Clean-Snapshot-20250507_095225-image.fits', '73MHz-Clean-Snapshot-20250507_095235-image.fits', '73MHz-Clean-Snapshot-20250507_095245-image.fits', '73MHz-Clean-Snapshot-20250507_095255-image.fits', '73MHz-Clean-Snapshot-20250507_095305-image.fits', '73MHz-Clean-Snapshot-20250507_095325-image.fits', '73MHz-Clean-Snapshot-20250507_095315-image.fits', '73MHz-Clean-Snapshot-20250507_095346-image.fits', '73MHz-Clean-Snapshot-20250507_095335-image.fits', '73MHz-Clean-Snapshot-20250507_095356-image.fits', '73MHz-Clean-Snapshot-20250507_095416-image.fits', '73MHz-Clean-Snapshot-20250507_095406-image.fits', '73MHz-Clean-Snapshot-20250507_095426-image.fits', '73MHz-Clean-Snapshot-20250507_095436-image.fits', '73MHz-Clean-Snapshot-20250507_095446-image.fits', '73MHz-Clean-Snapshot-20250507_095456-image.fits', '73MHz-Clean-Snapshot-20250507_095506-image.fits', '73MHz-Clean-Snapshot-20250507_095517-image.fits', '73MHz-Clean-Snapshot-20250507

In [18]:
if os.path.exists(csv_output_path):
    print("Found existing CSV. Loading progress...")
    combined_df = pd.read_csv(csv_output_path)
    # Track completed files by looking at the 'file_name' column in your CSV
    completed_files = set(combined_df['file_name'].unique())
else:
    print("No existing CSV found. Starting fresh.")
    combined_df = pd.DataFrame()
    completed_files = set()

# iterate through each file and complete the pybdsf process
for item in file_list_73:

  if item in completed_list:
    continue

  !funpack -O "{ryan_folder_path}OVRA-LWA-Uncompressed/{item}" "{folder_path}{item}.fz"
  img = bdsf.process_image(
      f"{ryan_folder_path}OVRA-LWA-Uncompressed/{item}",quiet = True
  )

  # Extract the position and flux of each source
  sources = getattr(img, "sources", [])
  ras = np.full(len(sources), np.nan)
  decs = np.full(len(sources), np.nan)
  fluxes = np.full(len(sources), np.nan)
  for source_ind, source in enumerate(sources):
    ras[source_ind] = source.posn_sky_centroid[0]
    decs[source_ind] = source.posn_sky_centroid[1]
    fluxes[source_ind] = source.total_flux

  # Only keep sources with non-nan values
  keep_sources_inds = np.where(np.isfinite(ras) & np.isfinite(decs) & np.isfinite(fluxes))
  ras = ras[keep_sources_inds]
  decs = decs[keep_sources_inds]
  fluxes = fluxes[keep_sources_inds]

  # Create a DataFrame from the filtered sources and include the filename
  filtered_df = pd.DataFrame({
      'ra': ras,
      'dec': decs,
      'total_flux': fluxes,
      'file_name': item
  })

  # Aggregate into the master dataframe
  combined_df = pd.concat([combined_df, filtered_df], ignore_index=True)

  # Save current status of the combined dataframe into a csv
  combined_df.to_csv(combined_csv_path, index=False)
  print(f"Updated CSV with sources from {item}. Total sources: {len(combined_df)}")

  completed_list.append(item)

  # Optional: Clean up uncompressed fits to save space
  !rm "{ryan_folder_path}OVRA-LWA-Uncompressed/{item}"

display(combined_df.head())

INFO:PyBDSF.Process:Processing /content/drive/MyDrive/Caltech SRC '26/OVRA-LWA-Uncompressed/73MHz-Clean-Snapshot-20250507_095214-image.fits
INFO:PyBDSF.Init:PyBDSF version 1.14.1
INFO:PyBDSF.Init:Non-default input parameters:
    filename             = "/content/drive/MyDrive/Caltech SRC '26/OVRA-LWA-Uncompressed/73MHz-Clean-Snapshot-20250507_095214-image.fits"
    quiet                = True
USERINFO:PyBDSF.Readfile:--> Opened '/content/drive/MyDrive/Caltech SRC '26/OVRA-LWA-Uncompressed/73MHz-Clean-Snapshot-20250507_095214-image.fits'
INFO:PyBDSF.Readfile:Original data shape of /content/drive/MyDrive/Caltech SRC '26/OVRA-LWA-Uncompressed/73MHz-Clean-Snapshot-20250507_095214-image.fits: (1, 1, 3122, 3122)
INFO:PyBDSF.Readfile:Final data shape (npol, nchan, x, y): (1, 1, 3122, 3122)
USERINFO:PyBDSF.Readimage:Image size .............................. : (3122, 3122) pixels
USERINFO:PyBDSF.Readimage:Number of channels ...................... : 1
USERINFO:PyBDSF.Readimage:Number of Stokes p

Updated CSV with sources from 73MHz-Clean-Snapshot-20250507_095214-image.fits. Total sources: 2132
Error: output file already exists:
 /content/drive/MyDrive/Caltech SRC '26/OVRA-LWA-Uncompressed/73MHz-Clean-Snapshot-20250507_095225-image.fits
 Input and output files are unchanged.


INFO:PyBDSF.Process:Processing /content/drive/MyDrive/Caltech SRC '26/OVRA-LWA-Uncompressed/73MHz-Clean-Snapshot-20250507_095225-image.fits
INFO:PyBDSF.Init:PyBDSF version 1.14.1
INFO:PyBDSF.Init:Non-default input parameters:
    filename             = "/content/drive/MyDrive/Caltech SRC '26/OVRA-LWA-Uncompressed/73MHz-Clean-Snapshot-20250507_095225-image.fits"
    quiet                = True
USERINFO:PyBDSF.Readfile:--> Opened '/content/drive/MyDrive/Caltech SRC '26/OVRA-LWA-Uncompressed/73MHz-Clean-Snapshot-20250507_095225-image.fits'
INFO:PyBDSF.Readfile:Original data shape of /content/drive/MyDrive/Caltech SRC '26/OVRA-LWA-Uncompressed/73MHz-Clean-Snapshot-20250507_095225-image.fits: (1, 1, 3122, 3122)
INFO:PyBDSF.Readfile:Final data shape (npol, nchan, x, y): (1, 1, 3122, 3122)
USERINFO:PyBDSF.Readimage:Image size .............................. : (3122, 3122) pixels
USERINFO:PyBDSF.Readimage:Number of channels ...................... : 1
USERINFO:PyBDSF.Readimage:Number of Stokes p

Updated CSV with sources from 73MHz-Clean-Snapshot-20250507_095225-image.fits. Total sources: 2662
Error: output file already exists:
 /content/drive/MyDrive/Caltech SRC '26/OVRA-LWA-Uncompressed/73MHz-Clean-Snapshot-20250507_095235-image.fits
 Input and output files are unchanged.


INFO:PyBDSF.Process:Processing /content/drive/MyDrive/Caltech SRC '26/OVRA-LWA-Uncompressed/73MHz-Clean-Snapshot-20250507_095235-image.fits
INFO:PyBDSF.Init:PyBDSF version 1.14.1
INFO:PyBDSF.Init:Non-default input parameters:
    filename             = "/content/drive/MyDrive/Caltech SRC '26/OVRA-LWA-Uncompressed/73MHz-Clean-Snapshot-20250507_095235-image.fits"
    quiet                = True
USERINFO:PyBDSF.Readfile:--> Opened '/content/drive/MyDrive/Caltech SRC '26/OVRA-LWA-Uncompressed/73MHz-Clean-Snapshot-20250507_095235-image.fits'
INFO:PyBDSF.Readfile:Original data shape of /content/drive/MyDrive/Caltech SRC '26/OVRA-LWA-Uncompressed/73MHz-Clean-Snapshot-20250507_095235-image.fits: (1, 1, 3122, 3122)
INFO:PyBDSF.Readfile:Final data shape (npol, nchan, x, y): (1, 1, 3122, 3122)
USERINFO:PyBDSF.Readimage:Image size .............................. : (3122, 3122) pixels
USERINFO:PyBDSF.Readimage:Number of channels ...................... : 1
USERINFO:PyBDSF.Readimage:Number of Stokes p

Updated CSV with sources from 73MHz-Clean-Snapshot-20250507_095235-image.fits. Total sources: 3205


INFO:PyBDSF.Process:Processing /content/drive/MyDrive/Caltech SRC '26/OVRA-LWA-Uncompressed/73MHz-Clean-Snapshot-20250507_095245-image.fits
INFO:PyBDSF.Init:PyBDSF version 1.14.1
INFO:PyBDSF.Init:Non-default input parameters:
    filename             = "/content/drive/MyDrive/Caltech SRC '26/OVRA-LWA-Uncompressed/73MHz-Clean-Snapshot-20250507_095245-image.fits"
    quiet                = True
USERINFO:PyBDSF.Readfile:--> Opened '/content/drive/MyDrive/Caltech SRC '26/OVRA-LWA-Uncompressed/73MHz-Clean-Snapshot-20250507_095245-image.fits'
INFO:PyBDSF.Readfile:Original data shape of /content/drive/MyDrive/Caltech SRC '26/OVRA-LWA-Uncompressed/73MHz-Clean-Snapshot-20250507_095245-image.fits: (1, 1, 3122, 3122)
INFO:PyBDSF.Readfile:Final data shape (npol, nchan, x, y): (1, 1, 3122, 3122)
USERINFO:PyBDSF.Readimage:Image size .............................. : (3122, 3122) pixels
USERINFO:PyBDSF.Readimage:Number of channels ...................... : 1
USERINFO:PyBDSF.Readimage:Number of Stokes p

Updated CSV with sources from 73MHz-Clean-Snapshot-20250507_095245-image.fits. Total sources: 3778


INFO:PyBDSF.Process:Processing /content/drive/MyDrive/Caltech SRC '26/OVRA-LWA-Uncompressed/73MHz-Clean-Snapshot-20250507_095255-image.fits
INFO:PyBDSF.Init:PyBDSF version 1.14.1
INFO:PyBDSF.Init:Non-default input parameters:
    filename             = "/content/drive/MyDrive/Caltech SRC '26/OVRA-LWA-Uncompressed/73MHz-Clean-Snapshot-20250507_095255-image.fits"
    quiet                = True
USERINFO:PyBDSF.Readfile:--> Opened '/content/drive/MyDrive/Caltech SRC '26/OVRA-LWA-Uncompressed/73MHz-Clean-Snapshot-20250507_095255-image.fits'
INFO:PyBDSF.Readfile:Original data shape of /content/drive/MyDrive/Caltech SRC '26/OVRA-LWA-Uncompressed/73MHz-Clean-Snapshot-20250507_095255-image.fits: (1, 1, 3122, 3122)
INFO:PyBDSF.Readfile:Final data shape (npol, nchan, x, y): (1, 1, 3122, 3122)
USERINFO:PyBDSF.Readimage:Image size .............................. : (3122, 3122) pixels
USERINFO:PyBDSF.Readimage:Number of channels ...................... : 1
USERINFO:PyBDSF.Readimage:Number of Stokes p

KeyboardInterrupt: 


# Leftover stuff from the original demo notebook below:

In [ ]:
print([filename for filename in datafiles if filename.endswith(".fits.fz")])  # Confirm that the datafiles were found

In [ ]:
# Plot an image
with fits.open(folder_path+file_name+".fz") as hdul:
    data = hdul[1].data.squeeze()
    header = hdul[1].header

wcs = WCS(header).celestial

norm = ImageNormalize(data, interval=ZScaleInterval())

fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, projection=wcs)
im = ax.imshow(data, origin="lower", cmap="Greys_r", norm=norm)

ax.set_xlabel("RA")
ax.set_ylabel("Dec")
ax.coords.grid(color="white", alpha=0.3, linestyle="solid")
fig.colorbar(im, ax=ax, label="Flux density (Jy/beam)")

plt.tight_layout()
plt.savefig("output.png", dpi=150)
plt.show()

In [ ]:
# Uncompress a data file (required to run BDSF)
# First path is the path to the uncompressed file (change to a folder in your local Drive)
# Second path is the path to the compressed file
!funpack -O "{folder_path}{file_name}" "{folder_path}{file_name}.fz"

In [ ]:
img = bdsf.process_image(
    f"{folder_path}{file_name}",
)

In [ ]:
# Plot pyBDSF output
img.show_fit()

In [ ]:
# Write the source information to a .csv file (optional)
img.write_catalog(outfile=f"{ryan_folder_path}{file_name}.csv",format="csv")

In [ ]:
# Extract the position and flux of each source
sources = getattr(img, "sources", [])
ras = np.full(len(sources), np.nan)
decs = np.full(len(sources), np.nan)
fluxes = np.full(len(sources), np.nan)
for source_ind, source in enumerate(sources):
  ras[source_ind] = source.posn_sky_centroid[0]
  decs[source_ind] = source.posn_sky_centroid[1]
  fluxes[source_ind] = source.total_flux

In [ ]:
# Note that some of the values are nan. I suspect this is because the sources
# are on or beyond the horizon (don't correspond to real astronomical sources).
print(ras[1000:1030])

In [ ]:
# Only keep sources with non-nan values
keep_sources_inds = np.where(np.isfinite(ras) & np.isfinite(decs) & np.isfinite(fluxes))
ras = ras[keep_sources_inds]
decs = decs[keep_sources_inds]
fluxes = fluxes[keep_sources_inds]

In [ ]:
# Create a DataFrame from the filtered sources
filtered_df = pd.DataFrame({
    'ra': ras,
    'dec': decs,
    'total_flux': fluxes
})

# Save the filtered data to a new CSV
output_path = f"{ryan_folder_path}{file_name}_filtered.csv"
filtered_df.to_csv(output_path, index=False)
print(f"Successfully exported {len(filtered_df)} sources to {output_path}")

In [ ]:
# Delete the uncompressed file to free space on your Drive
!rm "{folder_path}{file_name}"